# 1A pilot v5 — Can automatic diacritic restoration undo the damage? Are the partial-noise results seed-stable?

> **Tóm tắt (VI):** v5 **không chạy lại v4**. Nó dùng lại đúng dữ liệu, prompt, prefill và cách chấm của v4 (kiểm tra tự động), và chỉ chạy **7 điều kiện mới**:
> 1. `restored`: lấy bản **bỏ dấu hoàn toàn** (`strip`), cho một mô hình **khôi phục dấu tự động** thêm dấu lại, rồi mới đưa cho 16 LLM. Câu hỏi: khôi phục dấu lấy lại được bao nhiêu phần năng lực đã mất?
> 2. `partial25/50/75` với **2 seed mới** (`_s2`, `_s3`): chọn ngẫu nhiên *khác* những từ bị bỏ dấu, để chứng minh đường liều–đáp ứng không phụ thuộc một lần bốc thăm.
>
> Chạy trên **L4 GPU**, `Run all`, cho phép Drive. Cần secret `HF_TOKEN` (🔑, **không dán token vào cell**) để có Llama như v4. Kết quả ở `MyDrive/llm-research/1A-pilot-v5/`; v4 chỉ được đọc, không bị ghi đè.

**New conditions** (everything else is exactly v4):

| condition | what the 16 LLMs read |
|---|---|
| `restored` | the `strip` text after automatic diacritic restoration by `peterhung/vietnamese-accent-marker-xlm-roberta` (XLM-R large token classifier, Apache-2.0; it only adds diacritics, never adds or removes words) |
| `partial25_s2`, `partial50_s2`, `partial75_s2`, `partial25_s3`, `partial50_s3`, `partial75_s3` | same rates as v4's `partial25/50/75`, different random choice of words (v4's draw is seed 1) |

**Reuse from v4, checked before anything runs:** the same test and dev items (compared with v4's `data/test_items.csv`), the same answer prefill per (model, task) (from v4's `checks/validity.csv`), the calibration prior recomputed on the same dev items with the same prefill (compared with v4's), and a re-score of clean test items that must reproduce v4's answers. The validity gate is v4's: the same 74 pairs.

**Cost:** about 2.5–3.5 h on L4 (≈ 12–18 CU): the same number of scored conditions per item as v4 (7), without v4's format probe and free-generation check, plus downloading the 16 models again. Results are saved per (model, task), so after a disconnect just Run all again.

## Step 0 · Setup (as v4; v4's results folder is read only)

In [ ]:
import sys, os, re, json, ast, math, random, unicodedata, difflib, warnings, shutil, time, gc
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install -U transformers accelerate
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-research"
else:
    ROOT = os.environ.get("PILOT_ROOT", "outputs")
SMOKE = bool(os.environ.get("PILOT_SMOKE"))   # local test only: tiny samples, no validity gate
SRC = f"{ROOT}/1A-pilot-v4" + ("-smoke" if SMOKE else "")   # v4 results: read only
OUT = f"{ROOT}/1A-pilot-v5" + ("-smoke" if SMOKE else "")
for d in ["data", "preds", "figures", "tables", "checks"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
LOCAL_MODELS = json.loads(os.environ.get("PILOT_LOCAL_MODELS", "{}"))  # local test only: {"hub name": "folder"}

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="An input array is constant")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print("v4 results:", SRC, "| v5 results:", OUT)

CHECKS = []   # every automatic check, saved to checks/checks.csv
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "ok": bool(ok), "detail": str(detail)[:300]})
    print(("✅ " if ok else "❌ ") + name + (f" — {detail}" if detail else ""))
    pd.DataFrame(CHECKS).drop_duplicates("check", keep="last").to_csv(f"{OUT}/checks/checks.csv", index=False)
    return ok

def require(name, ok, detail=""):
    # a failed requirement stops the notebook: no numbers are better than wrong numbers
    if not check(name, ok, detail):
        raise RuntimeError(f"Stopped: {name}. {detail}")

In [ ]:
SEED = 42
TASKS = ["vsfc", "vsmec", "vihsd", "xnli", "vimmrc"]
CONDITIONS = ["clean", "partial25", "partial50", "partial75", "strip", "teencode", "mixed"]
NOISY = CONDITIONS[1:]
LEVEL = {"clean": 0, "partial25": 25, "partial50": 50, "partial75": 75, "strip": 100}

# items per class: test = the evaluation, dev = format choice and calibration (validation split, disjoint)
TEST_PER_CLASS = {"vsfc": 150, "vsmec": 35, "vihsd": 150, "xnli": 150, "vimmrc": 60}
DEV_PER_CLASS = {"vsfc": 30, "vsmec": 15, "vihsd": 30, "xnli": 30, "vimmrc": 20}
MIN_SYLLABLES = 3

MODEL_INFO = {  # hub name: (family, size in B, Vietnamese-adapted?)   small to large, so cheap ones finish first
    "Qwen/Qwen2.5-1.5B-Instruct": ("Qwen2.5", 1.5, False),
    "Qwen/Qwen3-1.7B": ("Qwen3", 1.7, False),
    "SeaLLMs/SeaLLMs-v3-1.5B-Chat": ("SeaLLMs-v3", 1.5, True),
    "bigscience/bloomz-1b7": ("bloomz", 1.7, False),
    "vilm/vinallama-2.7b-chat": ("VinaLLaMA", 2.7, True),
    "Qwen/Qwen2.5-3B-Instruct": ("Qwen2.5", 3.1, False),
    "bigscience/bloomz-3b": ("bloomz", 3.0, False),
    "Qwen/Qwen3-4B": ("Qwen3", 4.0, False),
    "microsoft/Phi-3.5-mini-instruct": ("Phi-3.5", 3.8, False),
    "Qwen/Qwen2.5-7B-Instruct": ("Qwen2.5", 7.6, False),
    "Qwen/Qwen3-8B": ("Qwen3", 8.2, False),
    "SeaLLMs/SeaLLMs-v3-7B-Chat": ("SeaLLMs-v3", 7.6, True),
    "vilm/vinallama-7b-chat": ("VinaLLaMA", 6.9, True),
    "bigscience/bloomz-7b1": ("bloomz", 7.1, False),
    "mistralai/Mistral-7B-Instruct-v0.3": ("Mistral", 7.2, False),
}
GATED_INFO = {"meta-llama/Llama-3.2-3B-Instruct": ("Llama-3", 3.2, False)}   # needs the HF_TOKEN secret

# Validity gate, fixed before any noisy result is seen (Step 5)
MIN_LETTER_MASS = 0.5     # median probability on the option letters, clean test
MIN_GEN_AGREEMENT = 0.7   # free greedy answer == scored answer
MIN_ABOVE_CHANCE = 0.10   # clean accuracy − chance, and its 95% CI lower bound must be > 0
B = 2000                  # bootstrap resamples
TOKEN_BUDGET = 4096       # padded tokens per forward batch (halved automatically on out-of-memory)
N_GEN = 40                # items per task × {clean, strip} for the free-generation check
if SMOKE:
    TEST_PER_CLASS = {t: 4 for t in TASKS}
    DEV_PER_CLASS = {t: 2 for t in TASKS}
    MIN_LETTER_MASS, MIN_GEN_AGREEMENT, MIN_ABOVE_CHANCE, B, N_GEN = 0.0, 0.0, -1.0, 200, 4

In [ ]:
# v5: the new conditions
SEEDS_EXTRA = [2, 3]                     # v4's partial draw is seed 1
SEED_CONDS = [f"partial{l}_s{k}" for k in SEEDS_EXTRA for l in (25, 50, 75)]
NEW_CONDS = SEED_CONDS + ["restored"]
RESTORER = "peterhung/vietnamese-accent-marker-xlm-roberta"
require("v4 results exist", os.path.exists(f"{SRC}/checks/validity.csv"), SRC)

In [ ]:
# Figure style: one colour per model family, the same in every figure
FAMILIES = ["Qwen2.5", "Qwen3", "SeaLLMs-v3", "VinaLLaMA", "bloomz", "Phi-3.5", "Mistral", "Llama-3"]
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FAM_COLOR = dict(zip(FAMILIES, PALETTE))
COND_COLOR = {"partial25": "#cde2fb", "partial50": "#86b6ef", "partial75": "#3987e5", "strip": "#0d366b",
              "teencode": "#eb6834", "mixed": "#6250d6"}
TASK_NAME = {"vsfc": "VSFC (sentiment)", "vsmec": "VSMEC (emotion)", "vihsd": "ViHSD (hate speech)",
             "xnli": "XNLI-vi (inference)", "vimmrc": "ViMMRC (reading)"}
SEQ = mpl.colors.LinearSegmentedColormap.from_list("blue", ["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984", "axes.labelcolor": "#2b2a28",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5, "pdf.fonttype": 42,
})
FIGS = []
def save(fig, name, caption):
    fig.savefig(f"{OUT}/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"{OUT}/figures/{name}.pdf", bbox_inches="tight")
    FIGS.append((name, caption))
    plt.show()

def save_table(df, name, caption, floatfmt="%.2f"):
    df.to_csv(f"{OUT}/tables/{name}.csv")
    try:
        df.to_latex(f"{OUT}/tables/{name}.tex", float_format=floatfmt, caption=caption, label=f"tab:{name}", na_rep="--")
    except Exception as e:
        print("latex export failed:", name, repr(e)[:100])

KEY = {}   # headline numbers → key_numbers.json and the conclusions

## Step 1 · Data (v4's code) and a check that it is v4's data

In [ ]:
PARQUET = "https://huggingface.co/datasets/{}/resolve/refs%2Fconvert%2Fparquet/{}/{}/0000.parquet"
LETTERS = "ABCDEFG"

TASK_SPECS = {
    "vsfc": dict(source=("uitnlp/vietnamese_students_feedback", "default"), label_col="sentiment", classes=[0, 1, 2],
                 labels=["tiêu cực", "trung tính", "tích cực"],
                 fields=[("sentence", "Nhận xét: ")],
                 intro="Đọc nhận xét sau của một sinh viên về môn học và cho biết thái độ của nhận xét.",
                 labels_en=["negative", "neutral", "positive"], fields_en=["Feedback: "],
                 intro_en="Read the following student feedback about a course and state its attitude."),
    "vsmec": dict(source=("ura-hcmut/UIT-VSMEC", "default"), label_col="Emotion",
                  classes=["Enjoyment", "Sadness", "Anger", "Fear", "Disgust", "Surprise", "Other"],
                  labels=["vui vẻ", "buồn bã", "tức giận", "sợ hãi", "ghê tởm", "ngạc nhiên", "khác"],
                  fields=[("Sentence", "Bình luận: ")],
                  intro="Đọc bình luận mạng xã hội sau và cho biết cảm xúc chính mà nó thể hiện.",
                  labels_en=["enjoyment", "sadness", "anger", "fear", "disgust", "surprise", "other"], fields_en=["Comment: "],
                  intro_en="Read the following social media comment and state the main emotion it expresses."),
    "vihsd": dict(source=("ura-hcmut/UIT-ViHSD", "default"), label_col="label_id", classes=[0, 1, 2],
                  labels=["bình thường (không có lời lẽ xấu)", "xúc phạm (lời lẽ thô tục nhưng không nhắm vào ai)",
                          "thù ghét (công kích, lăng mạ một người hay một nhóm người)"],
                  fields=[("free_text", "Bình luận: ")],
                  intro="Đọc bình luận mạng xã hội sau và phân loại nó.",
                  labels_en=["clean (no bad language)", "offensive (vulgar but not aimed at anyone)",
                             "hate (attacks or insults a person or a group)"], fields_en=["Comment: "],
                  intro_en="Read the following social media comment and classify it."),
    "xnli": dict(source=("facebook/xnli", "vi"), label_col="label", classes=[0, 1, 2],
                 labels=["kéo theo (giả thuyết chắc chắn đúng)", "trung lập (không xác định được)",
                         "mâu thuẫn (giả thuyết chắc chắn sai)"],
                 fields=[("premise", "Tiền đề: "), ("hypothesis", "Giả thuyết: ")],
                 intro="Cho một tiền đề và một giả thuyết. Hãy cho biết quan hệ giữa chúng.",
                 labels_en=["entailment (the hypothesis is definitely true)", "neutral (cannot be determined)",
                            "contradiction (the hypothesis is definitely false)"], fields_en=["Premise: ", "Hypothesis: "],
                 intro_en="Given a premise and a hypothesis, state the relation between them."),
    "vimmrc": dict(source=("ura-hcmut/ViMMRC", "default"), label_col="answer", classes=["A", "B", "C", "D"],
                   labels=None,   # the options are part of the item
                   fields=[("article", "Đoạn văn: "), ("question", "Câu hỏi: "), ("options", "Các lựa chọn:\n")],
                   intro="Đọc đoạn văn rồi trả lời câu hỏi trắc nghiệm.",
                   labels_en=None, fields_en=["Passage: ", "Question: ", "Options:\n"],
                   intro_en="Read the passage, then answer the multiple-choice question."),
}
K = {t: len(TASK_SPECS[t]["classes"]) for t in TASKS}
CHANCE = {t: 1 / K[t] for t in TASKS}

def nfc(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()

def strip_diacritics(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", s)

def syllable_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text) if any(ch.isalpha() for ch in m.group())]

def accent_share(text):
    w = [x for x in text.split() if any(c.isalpha() for c in x)]
    return sum(strip_diacritics(x) != x for x in w) / max(len(w), 1)

def load_split(task, split):
    spec = TASK_SPECS[task]
    raw = pd.read_parquet(PARQUET.format(*spec["source"], split))
    if task == "vimmrc":
        # options as one field, one option per line; the letters are part of the text the model reads
        raw["options"] = ["\n".join(f"{l}. {nfc(o)}" for l, o in zip("ABCD", ast.literal_eval(x) if isinstance(x, str) else x))
                          for x in raw.options]
    fields = [f for f, _ in spec["fields"]]
    for f in fields:
        raw[f] = raw[f].map(nfc) if f != "options" else raw[f].map(lambda s: "\n".join(nfc(x) for x in s.split("\n")))
    raw = raw[raw[spec["label_col"]].isin(spec["classes"])].copy()
    raw["key"] = raw[fields].agg("\n".join, axis=1)
    n0 = len(raw)
    dup = raw.key.map(raw.key.value_counts()) > 1
    noisy_src = raw[[f for f in fields if f != "options"]].agg(" ".join, axis=1)
    short = noisy_src.map(lambda s: len(syllable_spans(s))) < MIN_SYLLABLES
    no_diac = raw.key.map(strip_diacritics) == raw.key
    keep = raw[~dup & ~short & ~no_diac].copy()
    log = {"task": task, "split": split, "rows": n0, "duplicates": int(dup.sum()), "short": int(short.sum()),
           "no_diacritics": int(no_diac.sum()), "kept": len(keep)}
    return keep, log

def sample(df, task, per_class, seed):
    spec = TASK_SPECS[task]
    n_min = df[spec["label_col"]].value_counts().min()
    require(f"{task}: enough items per class", n_min >= per_class, f"smallest class {n_min} < {per_class}")
    s = df.groupby(spec["label_col"]).sample(n=per_class, random_state=seed).sample(frac=1, random_state=seed)
    s = s.reset_index(drop=True)
    s["label"] = s[spec["label_col"]].map(spec["classes"].index)
    s["item_id"] = range(len(s))
    return s

RAW, DEV, TEST, clean_log = {}, {}, {}, []
for t in TASKS:
    test, lt = load_split(t, "test")
    dev, ld = load_split(t, "validation")
    before = len(dev)
    dev = dev[~dev.key.isin(set(test.key))]
    ld["overlap_with_test"] = before - len(dev)
    clean_log += [lt, ld]
    RAW[t] = test
    TEST[t] = sample(test, t, TEST_PER_CLASS[t], SEED)
    DEV[t] = sample(dev, t, DEV_PER_CLASS[t], SEED + 1)
clean_log = pd.DataFrame(clean_log).set_index(["task", "split"])
save_table(clean_log, "table_data_cleaning", "Rows removed before sampling.", "%d")
display(clean_log)
for t in TASKS:
    require(f"{t}: dev and test share no text", not set(DEV[t].key) & set(TEST[t].key))
    require(f"{t}: test sample is balanced", TEST[t].label.value_counts().nunique() == 1)

## Step 2 · Noise (v4's code, plus the extra seeds)

`partialNN_sK` uses the same rule as `partialNN` with a different random draw: the random generator is seeded by the condition name, so `partial50_s2` picks other words than `partial50`.

In [ ]:
def partial_strip(text, rate, rng):
    words = text.split(" ")
    cand = [i for i, w in enumerate(words) if strip_diacritics(w) != w]
    chosen = set(rng.sample(cand, round(rate * len(cand))))
    return " ".join(strip_diacritics(w) if i in chosen else w for i, w in enumerate(words))

TEENCODE = {
    "giáo viên": "gv", "giảng viên": "gv", "sinh viên": "sv", "bài tập": "bt", "học sinh": "hs",
    "không": "ko", "được": "dc", "gì": "j", "với": "vs", "mình": "mk", "rồi": "r",
    "vậy": "v", "bạn": "bn", "quá": "wa", "nhiều": "nhìu", "người": "ng", "trước": "trc",
    "cũng": "cx", "thôi": "thui", "biết": "bít", "yêu": "iu", "giờ": "h",
    "nhưng": "nhg", "lắm": "lém", "thích": "thik", "những": "nhữg", "hiểu": "hỉu",
}
_teen_re = re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, TEENCODE), key=len, reverse=True)) + r")(?!\w)",
                      re.IGNORECASE)

def teencode(text):
    return _teen_re.sub(lambda m: TEENCODE[m.group(0).lower()], text)

def make_noisy(text, cond, item_key, field):
    rng = random.Random(f"{SEED}|{item_key}|{cond}|{field}")
    if cond == "clean":
        return text
    if cond == "strip":
        return strip_diacritics(text)
    if cond.startswith("partial"):
        return partial_strip(text, int(cond[len("partial"):].split("_")[0]) / 100, rng)   # partialNN or partialNN_sK
    if cond == "teencode":
        return teencode(text)
    if cond == "mixed":
        return partial_strip(teencode(text), 0.5, rng)
    raise ValueError(cond)

def expand(df, task, conds):
    fields = [f for f, _ in TASK_SPECS[task]["fields"]]
    rows = []
    for r in df.itertuples(index=False):
        r = r._asdict()
        for c in conds:
            row = {"task": task, "item_id": r["item_id"], "label": r["label"], "cond": c}
            for f in fields:
                noisy = r[f] if f == "options" and c == "clean" else make_noisy(r[f], c, r["key"], f)
                row[f], row[f + "_clean"] = noisy, r[f]
            row["text"] = "\n".join(row[f] for f in fields)
            row["clean"] = "\n".join(row[f + "_clean"] for f in fields)
            rows.append(row)
    return pd.DataFrame(rows)

DATA = {t: expand(TEST[t], t, CONDITIONS + SEED_CONDS) for t in TASKS}
DEVDATA = {t: expand(DEV[t], t, ["clean"]) for t in TASKS}
pd.concat(DATA.values())[["task", "item_id", "label", "cond", "text", "clean"]].to_csv(f"{OUT}/data/test_items.csv", index=False)
pd.concat(DEVDATA.values())[["task", "item_id", "label", "text"]].to_csv(f"{OUT}/data/dev_items.csv", index=False)

again = expand(TEST["vsfc"], "vsfc", CONDITIONS)
require("noise is deterministic", again.text.reset_index(drop=True).equals(
    DATA["vsfc"][DATA["vsfc"].cond.isin(CONDITIONS)].text.reset_index(drop=True)))
assert strip_diacritics("Tôi muốn đặt vé máy bay") == "Toi muon dat ve may bay"

rows = []
for t in TASKS:
    d = DATA[t]
    for c in NOISY:
        g = d[d.cond == c]
        row = {"task": t, "cond": c, "items changed": (g.text != g.clean).mean(),
               "syllables changed": np.mean([1 - difflib.SequenceMatcher(None, a.split(), b.split(), autojunk=False).ratio()
                                             for a, b in zip(g.clean, g.text)])}
        if c in LEVEL:
            row["only diacritics changed"] = (g.text.map(strip_diacritics) == g.clean.map(strip_diacritics)).mean()
        rows.append(row)
noise_tab = pd.DataFrame(rows)
noise_tab.to_csv(f"{OUT}/checks/noise.csv", index=False)
st = noise_tab[noise_tab.cond == "strip"]
require("strip changes every test item", (st["items changed"] == 1).all(), st.set_index("task")["items changed"].round(3).to_dict())
require("strip/partial change only diacritics", (noise_tab.dropna()["only diacritics changed"] == 1).all())
require("strip leaves no diacritic", all((DATA[t][DATA[t].cond == "strip"].text.map(strip_diacritics) ==
                                          DATA[t][DATA[t].cond == "strip"].text).all() for t in TASKS))
hit = tot = 0
for t in TASKS:
    g = DATA[t][DATA[t].cond == "partial50"]
    for r in g.itertuples():
        for f, _ in TASK_SPECS[t]["fields"]:
            a, b = getattr(r, f + "_clean").split(" "), getattr(r, f).split(" ")
            cand = [i for i, w in enumerate(a) if strip_diacritics(w) != w]
            hit += sum(a[i] != b[i] for i in cand)
            tot += len(cand)
check("partial50 strips ~50% of accented words", abs(hit / tot - 0.5) < 0.02, f"realized {hit / tot:.3f}")
display(noise_tab.pivot(index="task", columns="cond", values="items changed")[NOISY].round(2))
ex = DATA["vsfc"][DATA["vsfc"].item_id == 0].set_index("cond").text.reindex(CONDITIONS).to_frame("example (VSFC)")
save_table(ex, "table_noise_example", "One VSFC test item under every condition (original Vietnamese text).")
display(ex)

In [ ]:
# Same items, labels and noisy texts as v4 (v4's conditions)
old = pd.read_csv(f"{SRC}/data/test_items.csv", keep_default_na=False)
new = pd.concat(DATA.values())[["task", "item_id", "label", "cond", "text", "clean"]]
new = new[new.cond.isin(CONDITIONS)].reset_index(drop=True)
require("test items, labels and noisy texts identical to v4", len(old) == len(new) and (old.astype(str).values == new.astype(str).values).all(),
        f"{len(old)} vs {len(new)} rows")

# the extra seeds: same rate as v4's draw, different words
rows = []
for t in TASKS:
    for l in (25, 50, 75):
        base = DATA[t][DATA[t].cond == f"partial{l}"].text.values
        for k in SEEDS_EXTRA:
            g = DATA[t][DATA[t].cond == f"partial{l}_s{k}"]
            rows.append({"task": t, "level": l, "seed": k, "same text as seed 1": (g.text.values == base).mean(),
                         "only diacritics changed": (g.text.map(strip_diacritics) == g.clean.map(strip_diacritics)).mean()})
seed_tab = pd.DataFrame(rows)
seed_tab.to_csv(f"{OUT}/checks/seed_noise.csv", index=False)
require("extra seeds change only diacritics", (seed_tab["only diacritics changed"] == 1).all())
check("extra seeds pick different words than v4's draw", seed_tab["same text as seed 1"].mean() < 0.5,
      f"share of items identical to seed 1: {seed_tab['same text as seed 1'].mean():.2f}")
display(seed_tab.groupby(["task", "level"])["same text as seed 1"].mean().unstack().round(2))

## Step 2b · Automatic diacritic restoration of the `strip` texts

The restorer labels every word with a rewrite tag (e.g. `u-ữ`: replace `u` by `ữ`), following the model card. Our additions:
- each line is restored separately and words are split on single spaces, so line breaks (ViMMRC options) and spacing survive;
- long lines are cut into windows of 120 words (the model reads at most 512 sub-word tokens);
- the tag is applied to the lower-cased word and the original capitalisation is put back, so `Nhung` → `Những`.

**Restoration quality** is measured against the original text: share of words restored exactly, among all words and among the words that originally carried a diacritic, and share of items restored without any error. Restored texts are cached in `data/restored.csv`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from huggingface_hub import hf_hub_download
import torch
RCACHE = f"{OUT}/data/restored.csv"
WINDOW = 120

def fields_of(t):
    return [f for f, _ in TASK_SPECS[t]["fields"]]

def recase(src, low):
    # put the capitalisation of src back on its lower-cased rewrite (same length)
    return "".join(c.upper() if s.isupper() else c for s, c in zip(src, low)) if len(src) == len(low) else low

def restore_texts(texts):
    rtok = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(RESTORER, RESTORER), add_prefix_space=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    rmodel = AutoModelForTokenClassification.from_pretrained(LOCAL_MODELS.get(RESTORER, RESTORER)).to(dev).eval()
    tag_file = (os.path.join(LOCAL_MODELS[RESTORER], "selected_tags_names.txt") if RESTORER in LOCAL_MODELS
                else hf_hub_download(RESTORER, "selected_tags_names.txt"))
    tags = [l.strip() for l in open(tag_file, encoding="utf-8") if l.strip()]
    require("restorer: tag set as on the model card", len(tags) == rmodel.config.num_labels, f"{len(tags)} tags")
    tags = [tuple(x.split("-")) for x in tags]
    # windows of words: (text index, line index, start word) → words
    lines = [t.split("\n") for t in texts]
    jobs = []
    for i, ls in enumerate(lines):
        for j, line in enumerate(ls):
            words = line.split(" ")
            for s in range(0, len(words), WINDOW):
                jobs.append((i, j, s, words[s:s + WINDOW]))
    out_words = {}
    for b in range(0, len(jobs), 32):
        batch = jobs[b:b + 32]
        keep = [[w for w in ws if w] for _, _, _, ws in batch]
        enc = rtok([k if k else ["."] for k in keep], is_split_into_words=True, truncation=True, max_length=512,
                   padding=True, return_tensors="pt")
        with torch.no_grad():
            pred = rmodel(**{k: v.to(dev) for k, v in enc.items()}).logits.argmax(-1).cpu().numpy()
        for n, (i, j, s, ws) in enumerate(batch):
            labs = {}
            for p, w in enumerate(enc.word_ids(n)):
                if w is not None:
                    labs.setdefault(w, []).append(pred[n, p])
            res, w_i = [], 0
            for w in ws:
                if not w:
                    res.append(w)
                    continue
                low, new = w.lower(), None
                for li in labs.get(w_i, []):   # the first tag that changes the word, as on the model card
                    raw, vowel = tags[li]
                    if raw and raw in low:
                        new = low.replace(raw, vowel)
                        break
                res.append(recase(w, new) if new is not None else w)
                w_i += 1
            out_words[(i, j, s)] = res
    del rmodel
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    restored = []
    for i, ls in enumerate(lines):
        new_lines = []
        for j, line in enumerate(ls):
            words = line.split(" ")
            new_lines.append(" ".join(sum((out_words[(i, j, s)] for s in range(0, len(words), WINDOW)), [])))
        restored.append("\n".join(new_lines))
    return restored

# one text per (task, item, field), restored once and cached
units = [(t, r["item_id"], f, r[f]) for t in TASKS for r in DATA[t][DATA[t].cond == "strip"].to_dict("records") for f in fields_of(t)]
cached = pd.read_csv(RCACHE, keep_default_na=False) if os.path.exists(RCACHE) else None
if cached is not None and len(cached) == len(units) and (cached.stripped.values == np.array([u[3] for u in units])).all():
    print("restored texts read from", RCACHE)
else:
    t0 = time.time()
    rest = restore_texts([u[3] for u in units])
    cached = pd.DataFrame(units, columns=["task", "item_id", "field", "stripped"]).assign(restored=rest)
    cached.to_csv(RCACHE, index=False)
    print(f"restored {len(units)} fields in {time.time() - t0:.0f}s")
RMAP = {(r.task, r.item_id, r.field): r.restored for r in cached.itertuples()}
require("restoration keeps every word and line", all(len(a.split("\n")) == len(b.split("\n")) and len(a.split(" ")) == len(b.split(" "))
                                                       for a, b in zip(cached.stripped, cached.restored)))
require("restoration changes only diacritics", (cached.restored.map(strip_diacritics) == cached.stripped).all())

for t in TASKS:
    s = DATA[t][DATA[t].cond == "strip"].copy()
    s["cond"] = "restored"
    for f in fields_of(t):
        s[f] = [RMAP[(t, i, f)] for i in s.item_id]
    s["text"] = ["\n".join(r[f] for f in fields_of(t)) for r in s.to_dict("records")]
    DATA[t] = pd.concat([DATA[t], s], ignore_index=True)
pd.concat(DATA.values())[["task", "item_id", "label", "cond", "text", "clean"]].to_csv(f"{OUT}/data/test_items.csv", index=False)

# restoration quality against the original text
rows = []
for t in TASKS:
    g = DATA[t][DATA[t].cond == "restored"]
    tot = acc = acc_d = tot_d = exact = 0
    for r in g.to_dict("records"):
        a, b = r["clean"].split(), r["text"].split()
        same = [x == y for x, y in zip(a, b)]
        tot += len(a); acc += sum(same)
        had = [strip_diacritics(x) != x for x in a]
        tot_d += sum(had); acc_d += sum(s_ for s_, h in zip(same, had) if h)
        exact += all(same)
    rows.append({"task": t, "words correct": acc / tot, "accented words correct": acc_d / max(tot_d, 1), "items without error": exact / len(g)})
RQ = pd.DataFrame(rows).set_index("task")
save_table(RQ, "table_restoration_quality", "Automatic diacritic restoration of the fully stripped test texts: accuracy against the original.")
check("restoration accuracy plausible (≥ 85% of words; the model card reports 97%)", SMOKE or RQ["words correct"].mean() >= 0.85,
      f"mean {RQ['words correct'].mean():.3f}")
display(RQ.round(3))
KEY["restoration_quality"] = RQ.to_dict("index")
ex = DATA["vsfc"][DATA["vsfc"].item_id < 3].set_index(["item_id", "cond"]).text.unstack().reindex(columns=["clean", "strip", "restored"])
display(ex)

## Step 3 · Models and prompts: look before running (CPU)

For every model, using its tokenizer only:
1. **Prompt format**: the model's own chat template; VinaLLaMA-2.7b ships without one, so it gets the ChatML template from VinaLLaMA-7b (both model cards document ChatML); bloomz is not a chat model and gets a plain prompt. The exact prompt for one item is printed.
2. **Answer letters**: each option letter must be **one token** right after the prompt, and the letters must be different tokens. Checked for every prefill candidate.
3. **Length**: the longest prompt must fit the model's context.
4. **Tokenizer statistics** of the input fields *inside the prompt* (what the model actually reads): tokens per syllable, share of syllables split into several tokens.

A model that fails a check is listed and left out; nothing runs on a broken setup.

In [ ]:
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
MODEL_INFO.update(GATED_INFO if (HF_TOKEN or any(m in LOCAL_MODELS for m in GATED_INFO)) else {})
print("gated models:", "included" if any(m in MODEL_INFO for m in GATED_INFO) else "skipped (no HF_TOKEN secret)")
SHORT = {m: m.split("/")[1].replace("-Instruct", "").replace("-Chat", "").replace("-chat", "").replace("-mini-instruct", "-mini")
         .replace("-v0.3", "") for m in MODEL_INFO}

from transformers import AutoTokenizer
CHAT_TEMPLATE_FROM = {"vilm/vinallama-2.7b-chat": "vilm/vinallama-7b-chat"}   # documented ChatML, missing in the 2.7b tokenizer
PREFILLS = {"chat": ["Đáp án:", "", "**"], "plain": ["Đáp án:", ""]}

def load_tokenizer(name):
    tok = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(name, name), token=HF_TOKEN)
    source = "own chat template" if tok.chat_template else "plain prompt (not a chat model)"
    if name in CHAT_TEMPLATE_FROM:
        donor = CHAT_TEMPLATE_FROM[name]
        tok.chat_template = AutoTokenizer.from_pretrained(LOCAL_MODELS.get(donor, donor), token=HF_TOKEN).chat_template
        source = f"ChatML from {donor.split('/')[1]}"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok, source

def build_message(task, row):
    # returns the user message and the character span of every input field inside it
    spec = TASK_SPECS[task]
    parts = [(spec["intro"] + "\n\n", None)]
    for f, prefix in spec["fields"]:
        parts += [(prefix, None), (row[f], f), ("\n", None)]
    if spec["labels"]:
        parts.append(("\nCác lựa chọn:\n" + "\n".join(f"{L}. {o}" for L, o in zip(LETTERS, spec["labels"])) + "\n", None))
    letters = LETTERS[:K[task]]
    parts.append(("\nChỉ trả lời bằng một chữ cái: " + ", ".join(letters[:-1]) + " hoặc " + letters[-1] + ".", None))
    msg, spans = "", {}
    for text, f in parts:
        if f:
            spans[f] = (len(msg), len(msg) + len(text))
        msg += text
    return msg, spans

_WRAP = {}
def chat_wrapper(tok):
    # the chat template rendered once around a placeholder; the template copies the message verbatim (checked)
    if id(tok) not in _WRAP:
        ph = "<<<MESSAGE>>>"
        w = tok.apply_chat_template([{"role": "user", "content": ph}], tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
        assert w.count(ph) == 1, "chat template does not copy the message verbatim"
        test_msg = "Thử: tiếng Việt có dấu.\nDòng hai."
        full = tok.apply_chat_template([{"role": "user", "content": test_msg}], tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
        assert full == w.replace(ph, test_msg), "chat template changes the message"
        _WRAP[id(tok)] = w.split(ph)
    return _WRAP[id(tok)]

def build_prompt(tok, task, row, prefill):
    msg, spans = build_message(task, row)
    if tok.chat_template:
        head, tail = chat_wrapper(tok)
        off = len(head)
        prompt, add_special = head + msg + tail + prefill, False
    else:
        off, prompt, add_special = 0, msg + "\n" + prefill, True
    return prompt, add_special, {f: (a + off, b + off) for f, (a, b) in spans.items()}

def encode(tok, task, row, prefill):
    # token ids, the token positions of each field, and per-field tokenizer statistics
    prompt, add_special, spans = build_prompt(tok, task, row, prefill)
    enc = tok(prompt, add_special_tokens=add_special, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]
    pos, stats = {}, {}
    for f, (a, b) in spans.items():
        p = [i for i, (x, y) in enumerate(offs) if y > x and x < b and y > a and i > 0]
        sy = [(a + s, a + e) for s, e in syllable_spans(prompt[a:b])]
        # a syllable is "broken" if more than one token covers it; offsets are sorted, so walk them once
        tok_spans = [offs[i] for i in p]
        j, broken = 0, 0
        for s_, e_ in sy:
            while j < len(tok_spans) and tok_spans[j][1] <= s_:
                j += 1
            k, n = j, 0
            while k < len(tok_spans) and tok_spans[k][0] < e_:
                n += 1
                k += 1
            broken += n > 1
        pos[f] = p
        stats[f] = (len(p), len(sy), broken)
    return prompt, add_special, ids, pos, stats

_LETTERS = {}
def letter_ids(tok, prompt, add_special, ids, task):
    # for each option letter: token ids of " X" / "X" that are ONE token right after the prompt.
    # Every prompt of a task ends with the same text, so the answer is cached on that ending (and re-checked on its tokens).
    key = (id(tok), task, prompt[-80:], tuple(ids[-6:]))
    if key in _LETTERS:
        return _LETTERS[key]
    out = []
    for L in LETTERS[:K[task]]:
        v = []
        for s in (" " + L, L):
            full = tok(prompt + s, add_special_tokens=add_special)["input_ids"]
            if full[:len(ids)] == ids and len(full) == len(ids) + 1:
                v.append(full[-1])
        out.append(sorted(set(v)))
    _LETTERS[key] = out
    return out

def context_len(name, tok):
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(LOCAL_MODELS.get(name, name), token=HF_TOKEN)
    n = getattr(cfg, "max_position_embeddings", None) or getattr(cfg, "seq_length", None) or getattr(cfg, "n_positions", None)
    return n or 2048   # bloom uses ALiBi (no hard limit); 2048 is its training length

In [ ]:
TOK, setup_rows, TOKSTATS = {}, [], []
for name in MODEL_INFO:
    try:
        tok, source = load_tokenizer(name)
    except Exception as e:
        check(f"{SHORT[name]}: tokenizer loads", False, repr(e)[:150])
        continue
    kind = "chat" if tok.chat_template else "plain"
    ok, notes, longest = True, [], 0
    for t in TASKS:
        rows = DATA[t].to_dict("records")
        for pre in PREFILLS[kind]:
            r = rows[0]
            prompt, add, ids, pos, _ = encode(tok, t, r, pre)
            lid = letter_ids(tok, prompt, add, ids, t)
            firsts = [v[0] for v in lid if v]
            if not all(lid) or len(set(firsts)) != len(firsts):
                notes.append(f"{t}/prefill {pre!r}: letters not single distinct tokens")
                if pre == "Đáp án:":
                    ok = False
        for r in [x for x in rows if x["cond"] in ("clean", "strip", "restored")]:   # tokenizer statistics (v5: the conditions it needs)
            _, _, ids, _, stats = encode(tok, t, r, "Đáp án:")
            longest = max(longest, len(ids))
            TOKSTATS.append({"model": name, "task": t, "item_id": r["item_id"], "cond": r["cond"],
                             **{f"{k}_{f}": v for f, s in stats.items() for k, v in zip(["ntok", "nsyl", "nbroken"], s)}})
    ctx = context_len(name, tok)
    ok &= longest + 16 <= ctx
    TOK[name] = tok
    setup_rows.append({"model": SHORT[name], "family": MODEL_INFO[name][0], "size_B": MODEL_INFO[name][1],
                       "vi_adapted": MODEL_INFO[name][2], "prompt format": source, "longest prompt": longest,
                       "context": ctx, "ok": ok, "notes": "; ".join(notes)})
    check(f"{SHORT[name]}: letters are single tokens and the longest prompt fits", ok, "; ".join(notes) or f"{longest}/{ctx} tokens")
setup = pd.DataFrame(setup_rows).set_index("model")
save_table(setup.drop(columns="notes"), "table_models", "Models, prompt format, and prompt length.")
display(setup)
MODELS = [m for m in MODEL_INFO if m in TOK and setup.loc[SHORT[m], "ok"]]
print(len(MODELS), "models pass the setup checks")
TOKSTATS = pd.DataFrame(TOKSTATS)
TOKSTATS.to_csv(f"{OUT}/data/tokenizer_stats.csv", index=False)

In [ ]:
# v5 runs the same models as v4
v4_models = sorted(pd.read_csv(f"{SRC}/checks/validity.csv").model.unique())
missing = [m for m in v4_models if m not in MODELS]
check("every v4 model is available", not missing,
      f"missing {missing}" + (" — add the HF_TOKEN secret (🔑) for Llama and Run all again" if any("llama" in m for m in missing) else ""))

## Step 3 · Run the new conditions (GPU)

Per model, per task, with v4's prompt and scoring code:
1. the **prefill** v4 chose (read from v4's `checks/validity.csv`);
2. the **calibration prior** recomputed on v4's dev items with that prefill, and compared with the prior v4 stored;
3. a **reproduction check**: 24 clean test items re-scored; the answers must match v4's;
4. the **7 new conditions** scored in one forward pass each (answer + NLL), saved to `preds/`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from huggingface_hub import constants as hf_constants
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (torch.bfloat16 if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8
         else torch.float16 if DEVICE == "cuda" else torch.float32)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| dtype:", DTYPE)

def fname(name):
    return name.replace("/", "__")

def pad_batch(seqs, pad_id):
    Lm = max(map(len, seqs))
    ids = torch.full((len(seqs), Lm), pad_id)
    mask = torch.zeros((len(seqs), Lm), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

@torch.no_grad()
def forward_batch(model, tok, items):
    # items: dicts with ids, letters (list of token-id lists), pos (field → token positions)
    ids, mask = pad_batch([it["ids"] for it in items], tok.pad_token_id)
    logits = model(input_ids=ids, attention_mask=mask).logits
    last = logits[torch.arange(len(items), device=DEVICE), mask.sum(1) - 1].float().log_softmax(-1)
    out = []
    # gather log p(token_t | tokens < t) for every field token, in chunks to keep memory bounded
    pairs = [(b, p) for b, it in enumerate(items) for f in it["pos"] for p in it["pos"][f]]
    tok_lp = {}
    for i in range(0, len(pairs), 512):
        chunk = pairs[i:i + 512]
        bb = torch.tensor([b for b, _ in chunk], device=DEVICE)
        pp = torch.tensor([p for _, p in chunk], device=DEVICE)
        sel = logits[bb, pp - 1].float()
        lp = sel.gather(1, ids[bb, pp][:, None]).squeeze(1) - sel.logsumexp(-1)
        tok_lp.update(zip(chunk, lp.tolist()))
    del logits
    for b, it in enumerate(items):
        letter_lp = [torch.logsumexp(last[b, v], 0).item() for v in it["letters"]]
        nll = {f: -sum(tok_lp[(b, p)] for p in it["pos"][f]) for f in it["pos"]}
        out.append((letter_lp, nll))
    return out

def batches(items, budget):
    order = sorted(range(len(items)), key=lambda i: -len(items[i]["ids"]))
    cur, longest = [], 0
    for i in order:
        n = len(items[i]["ids"])
        if cur and max(longest, n) * (len(cur) + 1) > budget:
            yield cur
            cur, longest = [], 0
        cur.append(i)
        longest = max(longest, n)
    if cur:
        yield cur

def score(model, tok, task, rows, prefill, budget):
    items = []
    for r in rows:
        prompt, add, ids, pos, stats = encode(tok, task, r, prefill)
        items.append({"ids": ids, "pos": pos, "letters": letter_ids(tok, prompt, add, ids, task)})
    res = [None] * len(items)
    for idx in batches(items, budget):
        for i, o in zip(idx, forward_batch(model, tok, [items[i] for i in idx])):
            res[i] = o
    lp = np.array([r[0] for r in res])
    nll = pd.DataFrame([r[1] for r in res]).add_prefix("nll_")
    return lp, nll

def softmax_rows(lp):
    p = np.exp(lp - lp.max(1, keepdims=True))
    return p / p.sum(1, keepdims=True)

@torch.no_grad()
def generate(model, tok, task, rows, prefill, max_new_tokens=12):
    tok.padding_side = "left"
    out = []
    for i in range(0, len(rows), 8):
        enc = [encode(tok, task, r, prefill) for r in rows[i:i + 8]]
        Lm = max(len(e[2]) for e in enc)
        ids = torch.tensor([[tok.pad_token_id] * (Lm - len(e[2])) + e[2] for e in enc], device=DEVICE)
        mask = (torch.arange(Lm)[None, :] >= torch.tensor([Lm - len(e[2]) for e in enc])[:, None]).long().to(DEVICE)
        g = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        out += [tok.decode(x[Lm:], skip_special_tokens=True) for x in g]
    tok.padding_side = "right"
    return out

def parse_letter(text, task):
    m = re.search(r"\b([A-G])\b", text.replace("*", " ")[:40])
    return LETTERS.index(m.group(1)) if m and LETTERS.index(m.group(1)) < K[task] else -1

In [ ]:
V4VALID = pd.read_csv(f"{SRC}/checks/validity.csv", keep_default_na=False)
PREFILL = {(r.model, r.task): r.prefill for r in V4VALID.itertuples()}
N_REPRO = 4 if SMOKE else 24

def run_new(model, tok, name, task, budget):
    pre = PREFILL[(name, task)]
    lp_dev, _ = score(model, tok, task, DEVDATA[task].to_dict("records"), pre, budget)
    prior = softmax_rows(lp_dev).mean(0)
    pr = pd.read_csv(f"{SRC}/probe/{task}__{fname(name)}.csv", keep_default_na=False)
    prior_v4 = np.array(ast.literal_eval(pr[pr.chosen.astype(str) == "True"].prior.iloc[0]))
    v4p = pd.read_csv(f"{SRC}/preds/{task}__{fname(name)}.csv", keep_default_na=False)
    rep = DATA[task][DATA[task].cond == "clean"].head(N_REPRO)
    lp_rep, _ = score(model, tok, task, rep.to_dict("records"), pre, budget)
    ref = v4p[v4p.cond == "clean"].set_index("item_id").loc[rep.item_id]
    repro = {"prior_diff": float(np.abs(prior - prior_v4).max()),
             "answers_match": float((lp_rep.argmax(1) == ref.pred_raw.values).mean()),
             "lp_diff": float(np.abs(lp_rep - ref[[f"lp{k}" for k in range(K[task])]].to_numpy(float)).max())}
    rows = DATA[task][DATA[task].cond.isin(NEW_CONDS)]
    lp, nll = score(model, tok, task, rows.to_dict("records"), pre, budget)
    p = softmax_rows(lp)
    out = rows[["task", "item_id", "label", "cond"]].reset_index(drop=True)
    out["model"] = name
    for k in range(K[task]):
        out[f"lp{k}"] = lp[:, k]
    out["letter_mass"] = np.exp(lp).sum(1)
    out["pred_raw"] = p.argmax(1)
    out["pred"] = (np.log(p) - np.log(prior)).argmax(1)
    out = pd.concat([out, nll], axis=1)
    out["prefill"] = pre
    return out, repro

def done(name, task):
    return os.path.exists(f"{OUT}/preds/{task}__{fname(name)}.csv")

RUN_MODELS = [m for m in MODELS if m in v4_models]
todo = [(m, t) for m in RUN_MODELS for t in TASKS if not done(m, t)]
print(f"{len(todo)} (model, task) runs to do")
if todo and DEVICE != "cuda" and not SMOKE:
    raise RuntimeError("Model runs are missing and there is no GPU: Runtime → Change runtime type → L4 GPU")
REPRO_FILE = f"{OUT}/checks/reproduction.csv"
REPRO = pd.read_csv(REPRO_FILE).to_dict("records") if os.path.exists(REPRO_FILE) else []
failed = {}
for name in RUN_MODELS:
    tasks = [t for t in TASKS if not done(name, t)]
    if not tasks:
        continue
    t0 = time.time()
    try:
        tok = TOK[name]
        model = AutoModelForCausalLM.from_pretrained(LOCAL_MODELS.get(name, name), dtype=DTYPE, token=HF_TOKEN).to(DEVICE).eval()
    except Exception as e:
        failed[name] = repr(e)[:200]
        print("✗ could not load", name, failed[name])
        continue
    print(f"● {SHORT[name]} loaded in {time.time() - t0:.0f}s")
    for t in tasks:
        t1, budget = time.time(), TOKEN_BUDGET
        while True:
            try:
                out, repro = run_new(model, tok, name, t, budget)
                break
            except torch.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                budget //= 2
                print(f"   out of memory, token budget → {budget}")
                if budget < 512:
                    out = None
                    break
        if out is None:
            failed[f"{name}/{t}"] = "out of memory"
            continue
        out.to_csv(f"{OUT}/preds/{t}__{fname(name)}.csv", index=False)
        REPRO = [r for r in REPRO if (r["model"], r["task"]) != (name, t)] + [{"model": name, "task": t, **repro}]
        pd.DataFrame(REPRO).to_csv(REPRO_FILE, index=False)
        acc = (out.pred == out.label).groupby(out.cond).mean()
        print(f"   {t}: {time.time() - t1:.0f}s | restored {acc.get('restored', np.nan):.2f} | "
              f"prior Δ {repro['prior_diff']:.3f}, v4 answers reproduced {repro['answers_match']:.0%}")
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if IN_COLAB:   # 7B weights are ~15 GB; keep Colab's disk free
        shutil.rmtree(os.path.join(hf_constants.HF_HUB_CACHE, "models--" + name.replace("/", "--")), ignore_errors=True)
check("every (model, task) run finished", not failed and all(done(m, t) for m in RUN_MODELS for t in TASKS), failed or "")

## Step 4 · Audit: complete, consistent, and the same pipeline as v4

From here on, CPU only. The reproduction check compares v5's scoring with v4's on the same clean items: the answer must be the same (small differences in log-probabilities come from GPU batching). v4's validity gate decides which pairs enter the analysis.

In [ ]:
import glob
P4 = pd.concat([pd.read_csv(p, keep_default_na=False) for p in sorted(glob.glob(f"{SRC}/preds/*.csv"))], ignore_index=True)
P5 = pd.concat([pd.read_csv(f"{OUT}/preds/{t}__{fname(m)}.csv", keep_default_na=False) for m in RUN_MODELS for t in TASKS if done(m, t)],
               ignore_index=True)
P4 = P4[P4.model.isin(set(P5.model))]
P = pd.concat([P4, P5], ignore_index=True)
P["correct"] = (P.pred == P.label).astype(int)
ALL_CONDS = CONDITIONS + NEW_CONDS

rows = []
for (m, t), g in P5.groupby(["model", "task"]):
    lp = g[[f"lp{k}" for k in range(K[t])]].to_numpy(float)
    rows.append({"model": SHORT[m], "task": t, "rows": len(g), "expected": int(DATA[t].cond.isin(NEW_CONDS).sum()),
                 "duplicates": int(g.duplicated(["item_id", "cond"]).sum()), "finite": bool(np.isfinite(lp).all()),
                 "pred_raw_reproduced": float((lp.argmax(1) == g.pred_raw.values).mean())})
integ = pd.DataFrame(rows)
integ.to_csv(f"{OUT}/checks/integrity.csv", index=False)
bad = integ[(integ.rows != integ.expected) | (integ.duplicates > 0) | ~integ.finite | (integ.pred_raw_reproduced < 1)]
require("all new prediction files complete and consistent", bad.empty, bad.to_dict("records")[:3])

REP = pd.read_csv(REPRO_FILE)
display(REP.assign(model=REP.model.map(SHORT)).describe().round(3))
check("v4's answers reproduced on re-scored clean items (≥ 95%)", SMOKE or REP.answers_match.mean() >= 0.95,
      f"mean {REP.answers_match.mean():.3f}, worst {REP.answers_match.min():.2f}")
check("calibration prior matches v4's (max Δ < 0.02)", SMOKE or REP.prior_diff.max() < 0.02, f"max Δ {REP.prior_diff.max():.3f}")

OK = set(zip(V4VALID[V4VALID.valid.astype(str) == "True"].model, V4VALID[V4VALID.valid.astype(str) == "True"].task))
P["valid"] = [(m, t) in OK for m, t in zip(P.model, P.task)]
MODELS = [m for m in RUN_MODELS if m in set(P.model)]
print(f"{len(MODELS)} models; {len(OK & set(zip(P.model, P.task)))} valid (model, task) pairs (v4's gate)")

rng = np.random.default_rng(SEED)
N_ITEMS = {t: DATA[t].item_id.nunique() for t in TASKS}
IDX = {t: rng.integers(0, N_ITEMS[t], (B, N_ITEMS[t])) for t in TASKS}   # the same resamples as v4
ACC = {(m, t): g.pivot(index="item_id", columns="cond", values="correct").sort_index()[ALL_CONDS].to_numpy(float)
       for (m, t), g in P.groupby(["model", "task"])}
J = {c: i for i, c in enumerate(ALL_CONDS)}

def ci(a):
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) else (np.nan, np.nan)

def fmt_ci(v, lo, hi):
    return f"{v:.2f} [95% CI {lo:.2f}, {hi:.2f}]"

### Debug files: every item, its ground truth and every model's answer

One CSV per (model, task) in `debug/`, readable in Google Sheets: the exact input text of the item under each condition (v4's and v5's), the **ground truth** and the **predicted label as names** (calibrated = the answer used in the analysis, and uncalibrated), whether it is correct, the probability of every option, the letter mass, and — for the items of v4's free-generation check — the model's own free answer. Plus `debug/labels.csv`, the number/letter → label table.

In [ ]:
os.makedirs(f"{OUT}/debug", exist_ok=True)
NAMES = {t: TASK_SPECS[t]["labels_en"] or [f"option {L}" for L in LETTERS[:K[t]]] for t in TASKS}
pd.DataFrame([{"task": t, "number": k, "letter": LETTERS[k], "label": n} for t in TASKS for k, n in enumerate(NAMES[t])]
             ).to_csv(f"{OUT}/debug/labels.csv", index=False)
TEXT = pd.concat(DATA.values())[["task", "item_id", "cond", "text"]]
GEN4 = pd.concat([pd.read_csv(p, keep_default_na=False) for p in sorted(glob.glob(f"{SRC}/gen/*.csv"))], ignore_index=True)
order = {c: i for i, c in enumerate(ALL_CONDS)}
for (m, t), g in P.groupby(["model", "task"]):
    lp = g[[f"lp{k}" for k in range(K[t])]].to_numpy(float)
    p = np.exp(lp - lp.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    d = g[["task", "item_id", "cond"]].copy()
    d["ground_truth"] = g.label.map(lambda k: NAMES[t][k]).values
    d["predicted"] = g.pred.map(lambda k: NAMES[t][k]).values
    d["predicted_uncalibrated"] = g.pred_raw.map(lambda k: NAMES[t][k]).values
    d["correct"] = (g.pred == g.label).values
    for k in range(K[t]):
        d[f"p_{LETTERS[k]} ({NAMES[t][k]})"] = p[:, k].round(4)
    d["letter_mass"] = g.letter_mass.round(4).values
    d["valid_pair"] = (m, t) in OK
    gg = GEN4[(GEN4.model == m) & (GEN4.task == t)][["item_id", "cond", "answer"]].rename(columns={"answer": "free_answer (v4 check)"})
    d = d.merge(gg, on=["item_id", "cond"], how="left").merge(TEXT[TEXT.task == t], on=["task", "item_id", "cond"], how="left")
    d = d.sort_values(["item_id", "cond"], key=lambda s: s.map(order) if s.name == "cond" else s)
    d.to_csv(f"{OUT}/debug/{t}__{fname(m)}.csv", index=False)
n_dbg = len(glob.glob(f"{OUT}/debug/*__*.csv"))
require("a debug file for every (model, task)", n_dbg == P.groupby(["model", "task"]).ngroups, f"{n_dbg} files")
require("debug files: every row has its text", TEXT.merge(P[["task", "item_id", "cond"]], how="right").text.notna().all())
ex = pd.read_csv(f"{OUT}/debug/vsfc__{fname(MODELS[0])}.csv")
display(ex[ex.item_id == 0].drop(columns=["task"]))

## Step 5 · Does automatic restoration undo the damage?

For every valid (model, task):
- **normalised drop** under `strip` and under `restored` (as in v4: share of above-chance skill lost);
- **recovery** = (accuracy restored − accuracy strip) ÷ (accuracy clean − accuracy strip): the share of the lost accuracy that restoration brings back (1 = all of it, 0 = none). Only where stripping cost more than 5 points.

Means over valid models per task, 95% paired bootstrap CI over items (the same resamples as v4).

In [ ]:
def nd_boot(m, t, c):
    bs = ACC[(m, t)][IDX[t]].mean(1)
    den = bs[:, 0] - CHANCE[t]
    return (bs[:, 0] - bs[:, J[c]]) / np.where(den > 0.05, den, np.nan)

def rec_boot(m, t):
    bs = ACC[(m, t)][IDX[t]].mean(1)
    lost = bs[:, 0] - bs[:, J["strip"]]
    return (bs[:, J["restored"]] - bs[:, J["strip"]]) / np.where(lost > 0.05, lost, np.nan)

rows, per = [], []
for t in TASKS:
    ms = [m for m in MODELS if (m, t) in OK]
    if not ms:
        continue
    for m in ms:
        a = ACC[(m, t)].mean(0)
        lost = a[0] - a[J["strip"]]
        per.append({"model": SHORT[m], "task": t, "acc_clean": a[0], "acc_strip": a[J["strip"]], "acc_restored": a[J["restored"]],
                    "nd_strip": (a[0] - a[J["strip"]]) / (a[0] - CHANCE[t]), "nd_restored": (a[0] - a[J["restored"]]) / (a[0] - CHANCE[t]),
                    "recovery": (a[J["restored"]] - a[J["strip"]]) / lost if lost > 0.05 else np.nan})
    pt = pd.DataFrame([r for r in per if r["task"] == t])
    r = {"task": t, "models": len(ms)}
    for name, b in [("nd_strip", np.nanmean(np.stack([nd_boot(m, t, "strip") for m in ms]), 0)),
                    ("nd_restored", np.nanmean(np.stack([nd_boot(m, t, "restored") for m in ms]), 0)),
                    ("recovery", np.nanmean(np.stack([rec_boot(m, t) for m in ms]), 0))]:
        r[name] = pt[name].mean()
        r[name + "_lo"], r[name + "_hi"] = ci(b)
    rows.append(r)
REST = pd.DataFrame(rows).set_index("task")
PER = pd.DataFrame(per)
save_table(REST, "table_restoration_effect", "Normalised drop under strip and after automatic restoration, and the share of the lost accuracy recovered (valid models, 95% CI).")
save_table(PER.set_index(["model", "task"]), "table_restoration_per_model", "Per valid (model, task): accuracy clean / strip / restored, normalised drops and recovery.")
display(REST.round(2))
KEY["restoration_effect"] = REST.to_dict("index")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
x = np.arange(len(REST))
for j, (c, col, lab) in enumerate([("nd_strip", COND_COLOR["strip"], "strip"), ("nd_restored", "#1baf7a", "strip → restored")]):
    ax.bar(x + (j - 0.5) * 0.36, REST[c], 0.34, color=col, label=lab,
           yerr=[REST[c] - REST[c + "_lo"], REST[c + "_hi"] - REST[c]], error_kw=dict(lw=1, ecolor="#52514e"))
ax.axhline(0, color="#8a8984", lw=1)
ax.set(xticks=x, xticklabels=REST.index, ylabel="normalised drop (0 = none, 1 = chance)",
       title="a. Loss without diacritics, and after restoring them")
ax.legend(fontsize=7)
ax = axes[1]
for i, t in enumerate(REST.index):
    g = PER[(PER.task == t)].dropna(subset=["recovery"])
    ax.scatter(np.full(len(g), i) + np.linspace(-0.15, 0.15, len(g)), g.recovery, s=16, color="#86b6ef", zorder=2)
    ax.errorbar(i, REST.loc[t, "recovery"], yerr=[[REST.loc[t, "recovery"] - REST.loc[t, "recovery_lo"]],
                [REST.loc[t, "recovery_hi"] - REST.loc[t, "recovery"]]], fmt="o", color="#0d366b", ms=6, lw=1.5, zorder=3)
ax.axhline(1, color="#8a8984", lw=1, ls=":")
ax.axhline(0, color="#8a8984", lw=1)
ax.set(xticks=range(len(REST)), xticklabels=REST.index, ylabel="share of lost accuracy recovered",
       title="b. Recovery (dots: models; dark: mean, 95% CI)")
plt.tight_layout()
save(fig, "fig11_restoration", "a. Mean normalised drop under full stripping and after automatic diacritic restoration (valid models, 95% CI); "
     "b. share of the accuracy lost to stripping that restoration recovers.")

## Step 6 · Are the partial-noise results stable across seeds?

The v4 draw (seed 1) and the two new draws (seeds 2, 3) of `partial25/50/75`. Per task: mean normalised drop over valid models for each seed, the spread across seeds, and the **seed-averaged** dose–response (accuracy averaged over the three draws per item, then the v4 bootstrap). Dose–response monotonicity is re-counted on the seed-averaged accuracies.

In [ ]:
SEEDED = {l: [f"partial{l}", f"partial{l}_s2", f"partial{l}_s3"] for l in (25, 50, 75)}
rows = []
for t in TASKS:
    ms = [m for m in MODELS if (m, t) in OK]
    for l, cs in SEEDED.items():
        nds = [np.nanmean([((ACC[(m, t)][:, 0].mean() - ACC[(m, t)][:, J[c]].mean()) / (ACC[(m, t)][:, 0].mean() - CHANCE[t]))
                           for m in ms]) for c in cs]
        avg = np.nanmean(np.stack([np.nanmean(np.stack([nd_boot(m, t, c) for c in cs]), 0) for m in ms]), 0)
        lo, hi = ci(avg)
        rows.append({"task": t, "level": l, "seed 1 (v4)": nds[0], "seed 2": nds[1], "seed 3": nds[2],
                     "spread (max − min)": max(nds) - min(nds), "mean of seeds": np.mean(nds), "lo": lo, "hi": hi})
SEEDTAB = pd.DataFrame(rows).set_index(["task", "level"])
save_table(SEEDTAB, "table_seed_stability", "Mean normalised drop over valid models for three random draws of partial stripping; seed-averaged 95% CI.")
display(SEEDTAB.round(3))
KEY["seed_spread_max"] = float(SEEDTAB["spread (max − min)"].max())
KEY["seed_spread_mean"] = float(SEEDTAB["spread (max − min)"].mean())

mono = []
for (m, t) in sorted(OK & set(zip(P.model, P.task))):
    a = ACC[(m, t)].mean(0)
    curve = [a[0]] + [np.mean([a[J[c]] for c in SEEDED[l]]) for l in (25, 50, 75)] + [a[J["strip"]]]
    mono.append(np.all(np.diff(curve) <= 0.02))
KEY["monotone_seed_avg"] = float(np.mean(mono))
print(f"dose-response (near-)monotone with seed-averaged accuracies: {np.mean(mono):.0%} of valid pairs")

fig, axes = plt.subplots(1, len(TASKS), figsize=(3.1 * len(TASKS), 3.2), sharey=True)
for ax, (i, t) in zip(axes, enumerate(TASKS)):
    g = SEEDTAB.loc[t] if t in SEEDTAB.index.get_level_values(0) else None
    if g is None or g.empty:
        continue
    strip_nd = np.nanmean([(ACC[(m, t)][:, 0].mean() - ACC[(m, t)][:, J["strip"]].mean()) / (ACC[(m, t)][:, 0].mean() - CHANCE[t])
                           for m in MODELS if (m, t) in OK])
    xs = [0, 25, 50, 75, 100]
    for k, col in [("seed 1 (v4)", "#cde2fb"), ("seed 2", "#86b6ef"), ("seed 3", "#3987e5")]:
        ax.plot(xs, [0] + list(g[k]) + [strip_nd], color=col, lw=1.2, marker="o", ms=3, label=k)
    ax.fill_between(xs[1:4], g.lo, g.hi, color=PALETTE[i], alpha=0.15, lw=0)
    ax.plot(xs, [0] + list(g["mean of seeds"]) + [strip_nd], color=PALETTE[i], lw=2.2, label="mean of seeds")
    ax.set(title=TASK_NAME[t], xlabel="% of accented words stripped", xticks=xs)
axes[0].set_ylabel("normalised drop")
axes[0].legend(fontsize=6)
plt.tight_layout()
save(fig, "fig12_seed_stability", "Dose-response for three random draws of partial stripping (thin) and their mean with a 95% CI (thick), valid models.")

## Step 7 · Conclusions (auto-generated)

In [ ]:
L = ["# Pilot 1A v5 — restoration and seed stability (auto-generated)", ""]
rq = KEY["restoration_quality"]
L += ["## Automatic diacritic restoration",
      f"- Restorer: `{RESTORER}`. Words restored exactly: " + ", ".join(f"{t} {v['words correct']:.1%}" for t, v in rq.items())
      + "; items without any error: " + ", ".join(f"{t} {v['items without error']:.0%}" for t, v in rq.items()) + "."]
for t, r in REST.iterrows():
    L.append(f"- **{TASK_NAME[t]}** ({int(r.models)} valid models): normalised drop {fmt_ci(r.nd_strip, r.nd_strip_lo, r.nd_strip_hi)} without diacritics, "
             f"{fmt_ci(r.nd_restored, r.nd_restored_lo, r.nd_restored_hi)} after restoration; restoration recovers "
             f"{fmt_ci(r.recovery, r.recovery_lo, r.recovery_hi)} of the lost accuracy.")
L.append(f"- Across tasks, restoration recovers on average {REST.recovery.mean():.0%} of the accuracy lost to stripping; "
         f"the residual loss is {REST.nd_restored.mean():.2f} of the above-chance skill (vs {REST.nd_strip.mean():.2f} without diacritics).")
L += ["", "## Seed stability of partial stripping",
      f"- Across tasks and levels, the mean normalised drop differs between the three random draws by {KEY['seed_spread_mean']:.3f} on average "
      f"(largest spread {KEY['seed_spread_max']:.3f}).",
      f"- With seed-averaged accuracies the dose-response is (near-)monotone in {KEY['monotone_seed_avg']:.0%} of valid pairs."]
L += ["", "## Notes for the paper",
      "- Restoration is a pre-processing baseline, not a property of the LLMs: report the restorer and its accuracy with the result.",
      "- The restorer was trained on general Vietnamese text; social-media text (ViHSD) and reading passages differ in style, "
      "which shows in the per-task restoration accuracy.",
      "- Same items, prompts, prefills and validity gate as v4 (checked in Steps 2–4)."]
TEXT = "\n".join(L)
open(f"{OUT}/conclusions.md", "w").write(TEXT)
display(Markdown(TEXT))

In [ ]:
def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(f"{float(o):.4g}")   # 4 significant digits: keeps small p-values
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o
json.dump(jsonable(KEY), open(f"{OUT}/key_numbers.json", "w"), ensure_ascii=False, indent=1)
pd.DataFrame(FIGS, columns=["figure", "caption"]).to_csv(f"{OUT}/figures/captions.csv", index=False)
print("checks:", sum(c["ok"] for c in CHECKS), "passed,", sum(not c["ok"] for c in CHECKS), "failed")
print("figures:")
for n, c in FIGS:
    print(f"  {n}: {c}")
print("\nEverything is in", OUT, "— when done: Runtime → Disconnect and delete runtime.")